In [6]:
class NdLinear(nn.Module):
    def __init__(self, input_dims: tuple, hidden_size: tuple, transform_outer=True):
        """
        NdLinear: A PyTorch layer for projecting tensors into multi-space representations.

        Unlike conventional embedding layers that map into a single vector space, NdLinear
        transforms tensors across a collection of vector spaces, capturing multivariate structure
        and topical information that standard deep learning architectures typically lose.

        Args:
            input_dims (tuple): Shape of input tensor (excluding batch dimension).
            hidden_size (tuple): Target hidden dimensions after transformation.
        """
        super(NdLinear, self).__init__()

        if len(input_dims) != len(hidden_size):
            raise Exception("Input shape and hidden shape do not match.")

        self.input_dims = input_dims
        self.hidden_size = hidden_size
        self.num_layers = len(input_dims)  # Must match since dims are equal
        self.transform_outer = transform_outer

        # Define transformation layers per dimension
        self.align_layers = nn.ModuleList([
            nn.Linear(input_dims[i], hidden_size[i]) for i in range(self.num_layers)
        ])


    def forward(self, X):
        """
        Forward pass to project input tensor into a new multi-space representation.
        - Incrementally transposes, flattens, applies linear layers, and restores shape.

        Expected Input Shape: [batch_size, *input_dims]
        Output Shape: [batch_size, *hidden_size]

        Args:
            X (torch.Tensor): Input tensor with shape [batch_size, *input_dims]

        Returns:
            torch.Tensor: Output tensor with shape [batch_size, *hidden_size]
        """
        num_transforms = self.num_layers  # Number of transformations

        # Define iteration order
        # transform_indices = range(num_transforms) if transform_outer else reversed(range(num_transforms))

        for i in range(num_transforms):
            if self.transform_outer:
                layer = self.align_layers[i]
                transpose_dim = i + 1
            else:
                layer = self.align_layers[num_transforms - (i+1)]
                transpose_dim = num_transforms - i

            # Transpose the selected dimension to the last position

            X = torch.transpose(X, transpose_dim, num_transforms).contiguous()

            # Store original shape before transformation
            X_size = X.shape[:-1]

            # Flatten everything except the last dimension
            X = X.view(-1, X.shape[-1])

            # Apply transformation
            X = layer(X)

            # Reshape back to the original spatial structure (with new embedding dim)
            X = X.view(*X_size, X.shape[-1])

            # Transpose the dimension back to its original position
            X = torch.transpose(X, transpose_dim, num_transforms).contiguous()

        return X

## After reviewing the NdLinear code on GitHub and gaining a better understanding of how nn.Linear works in PyTorch, I decided to create this project to explore how NdLinear functions and how it differs from PyTorch’s built-in nn.Linear layer. As someone with limited experience in deep learning, I saw this project as an opportunity to build a stronger foundation and gain deeper insight into how neural network layers operate behind the scenes. I believe this understanding will be valuable for future projects.


###**1. Limitations of nn.Linear: Transforming Only the Last Dimension**

To start, nn.Linear can operate on both vectors and matrices. However, when applied to tensors with more than two dimensions, nn.Linear only transforms the innermost (last) dimension. The other dimensions remain unchanged in the output.


For example:


In [7]:
import torch
from torch import nn

In [8]:
X1 = torch.randn(32, 28, 6)
nnlinear_model1 = nn.Linear(in_features = 6, out_features = 2)
output_X1 = nnlinear_model1(X1)

In [9]:
output_X1.shape

torch.Size([32, 28, 2])

As we can see, the output tensor X1 only differs from the original X1 in its last dimension. This means we cannot use nn.Linear to transform multiple dimensions simultaneously; It's not possible to directly transform a tensor of shape [32, 28, 6] into [32, 10, 2] using a single nn.Linear layer.

However, NdLinear provides greater flexibility by allowing transformations across multiple dimensions (not just the innermost one). This makes it especially useful for high dimensional data where features are distributed across several axes.


Here's an example to illustrate this behavior:

In [10]:
X2 = torch.randn(64, 32, 28, 6)
ndlinear_model1 = NdLinear(input_dims = (32, 28, 6), hidden_size = (14, 20, 3)) # hidden_size is the output dimension excluding batch dimension
output_X2 = ndlinear_model1(X2)

In [11]:
output_X2.shape

torch.Size([64, 14, 20, 3])

###**2. Shared vs. Separate Weights**

Another limitation of the standard nn.Linear layer is that when it's given a tensor with more than two dimensions, it first flattens all dimensions except the last into a single batch-like dimension. It then applies the same weight matrix only to the last dimension, ignoring any structural relationships between the other dimensions. This can lead to a loss of information and potentially reduce model accuracy.

I created a LinearRegressionModel capable of accepting multi-dimensional tensors as input and transforming them into a desired output shape. The model and its training routine follow the core principles of PyTorch’s nn.Linear, including parameter updates using PyTorch’s built in optimizer functions. However, to deepen my understanding of how weights and biases are updated through loss computation (using the squared loss) and gradient descent, I chose to implement the parameter update process manually instead of using PyTorch’s automatic optimization.

In [12]:
class LinearRegressionModel(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.linear_model = nn.Linear(input_features, output_features)

        # Set initial weights and bias to zero
        with torch.no_grad():
            self.linear_model.weight.zero_()
            self.linear_model.bias.zero_()

    def forward(self, x):
        return self.linear_model(x)

In [13]:
def train(model, X, y, num_epochs, learning_rate=0.01):

    X = X.float()
    # Flatten all dimensions except the last one
    X = X.view(-1, X.shape[-1])

    for i in range(num_epochs):
        for z in range(len(y)):
            inp = X[z]
            y_actual = y[z]

            # Forward pass
            y_predict = model(inp)

            # Compute gradients manually using squared error
            error = y_actual - y_predict
            with torch.no_grad():
                for s in range(inp.shape[0]):
                    grad_w = 2 * error * (-inp[s])

                    model.linear_model.weight[0, s] -= (learning_rate * grad_w).item()

                grad_b = -2 * error * 1.0
                model.linear_model.bias[0] -= learning_rate * grad_b.item()

        # Evaluate after each epoch
        y_predict_all = model(X)
        squared_error = ((y_predict_all - y) ** 2).mean().item()
        print(f"Epoch {i+1}/{num_epochs}: Squared Error Loss = {squared_error:.4f}")
        print(f"Model weights: {model.linear_model.weight.data}; Bias: {model.linear_model.bias.data}")


In [14]:
X3 = torch.randn(10, 6, 3)
y_actual = torch.randn(60, 1)
model3 = LinearRegressionModel(3, 1)
train(model3, X3, y_actual, 10)


Epoch 1/10: Squared Error Loss = 0.7197
Model weights: tensor([[0.0824, 0.0521, 0.0050]]); Bias: tensor([0.0696])
Epoch 2/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1019, 0.0656, 0.0088]]); Bias: tensor([0.0889])
Epoch 3/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1062, 0.0694, 0.0106]]); Bias: tensor([0.0944])
Epoch 4/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1069, 0.0707, 0.0115]]); Bias: tensor([0.0961])
Epoch 5/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1069, 0.0712, 0.0118]]); Bias: tensor([0.0966])
Epoch 6/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1068, 0.0714, 0.0120]]); Bias: tensor([0.0968])
Epoch 7/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1067, 0.0715, 0.0121]]); Bias: tensor([0.0969])
Epoch 8/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1067, 0.0716, 0.0121]]); Bias: tensor([0.0969])
Epoch 9/10: Squared Error Loss = 0.7189
Model weights: tensor([[0.1067, 0.0716, 0.0121]]

We can see that in order to transform a multi-dimensional input matrix into the desired output, we must first flatten the matrix from shape [10, 6, 3] to [60, 3]. This flattening process means that parameter adjustments are only made to the last dimension, based on the actual output values. This is a common approach but limits the model’s ability to capture relationships between the other dimensions which reduces accuracy.



However, NdLinear offers more flexibility and preserves the original structure of the input, applying nn.Linear and updating parameters separately for each dimension instead of flattening everything into 2D like what PyTorch nn.Linear does. The transformation for each dimension is applied by first transposing the input tensor and move the relevant dimension is to the last position. The tensor is then flattened (except for the last dimension) to make it compatible for the nn.Linear transformation. After applying the linear transformation, the tensor matrix is transposed back to its original order with correpsonding updated dimension.

This approach allows NdLinear to update parameters for each dimension independently which maintains the structure of the input.

**Additionally, I noticed that NdLinear can transform a matrix either from outer to inner dimensions or from inner to outer by setting the transform_outer parameter to True or False.**

In [29]:
X4 = torch.randn(2, 28, 20)
ndlinear_model2 = NdLinear(input_dims = (28, 20), hidden_size = (14, 5))

# TRANSFORM MATRIX FROM OUTER TO INNER
output_X4 = ndlinear_model2(X4)
output_X4.shape

torch.Size([2, 14, 5])

In [30]:
# Display each nn.Linear layer's weights and biases

for i, layer in enumerate(ndlinear_model2.align_layers):
    print(f"Layer {i + 1}: Transforming dimension {i}")
    print(f"linear model is: {ndlinear_model2.align_layers[i]}")
    print(f"Weight parameters shape: {layer.weight.shape}")
    print(f"Bias parameters shape: {layer.bias.shape}\n")

Layer 1: Transforming dimension 0
linear model is: Linear(in_features=28, out_features=14, bias=True)
Weight parameters shape: torch.Size([14, 28])
Bias parameters shape: torch.Size([14])

Layer 2: Transforming dimension 1
linear model is: Linear(in_features=20, out_features=5, bias=True)
Weight parameters shape: torch.Size([5, 20])
Bias parameters shape: torch.Size([5])



In [31]:
# TRANSFORM MATRIX FROM INNER TO OUTER
ndlinear_model3 = NdLinear(input_dims = (28, 20), hidden_size = (14, 5), transform_outer=False)

output2_X4 = ndlinear_model3(X4)
output2_X4.shape # The outputs have the same shape, but the transformation orders are different

torch.Size([2, 14, 5])

In [37]:
# The order in which dimensions are transformed

for i in reversed(range(len(ndlinear_model3.align_layers))):
    layer = ndlinear_model3.align_layers[i]
    print(f"Layer {i + 1}: Transforming dimension {i}")
    print(f"Linear model is: {layer}")
    print(f"Weight parameters shape: {layer.weight.shape}")
    print(f"Bias parameters shape: {layer.bias.shape}\n")

Layer 2: Transforming dimension 1
Linear model is: Linear(in_features=20, out_features=5, bias=True)
Weight parameters shape: torch.Size([5, 20])
Bias parameters shape: torch.Size([5])

Layer 1: Transforming dimension 0
Linear model is: Linear(in_features=28, out_features=14, bias=True)
Weight parameters shape: torch.Size([14, 28])
Bias parameters shape: torch.Size([14])



If only using nn.Linear to perform the same procedure, we would have to flatten the input matrix to a 2D matrix which will destryo potentialk strutucre of the matrix. Here's an example to illustrate:

In [ ]:
X4_flattened = X4.view(2, -1) # we keep the first/outer dimension the same because the first dimension is usually batch or sample size
nnlinear_model2 = nn.Linear(X4_flattened.shape[-1], (14*5))
nnlinear_model2(X4_flattened).shape

torch.Size([2, 70])

In [ ]:
print(f"Weight parameters shape: {nnlinear_model2.weight.shape}")
print(f"Bias parameters shape: {nnlinear_model2.bias.shape}\n")

Weight parameters shape: torch.Size([70, 560])
Bias parameters shape: torch.Size([70])





---


#### **Further Observations**

Something else I've noticed about the NdLinear class is that the outer dimension of the input must correspond to the batch size, and it isn't passed in as input_dims. This means the input matrix will always have an extra dimension compared to input_dims. If the input matrix's shape matches the input_dims passed into NdLinear, the forward method won't transform dimensions before dimension 1, ignoring dimension 0 (which typically represents the batch size). Without applying a transformation to match the output size, num_transforms will also be out of range when performing the transpose.

Here's an example to illustrate:

In [44]:
X5 = torch.randn(10, 8)
ndlinear_model4 = NdLinear(input_dims = (10, 8), hidden_size = (5, 4))

# This will cause an Error!
ndlinear_model4(X5)


IndexError: Dimension out of range (expected to be in range of [-2, 1], but got 2)

Thank you so much for taking the time to go through this notebook. Since I’m still quite new to the machine learning field and learning how to understand and apply various deep learning models, I created this project to help me grasp the fundamentals of these two methods and their implementations. I’m excited to keep exploring NdLinear and hope to apply it to real deep learning projects in the future!

